# Video Tokenizer Training (Spacetime Vector Quantized Variational Autoencoder)

In [1]:
import torch 
import lpips

from torch.utils.data import DataLoader

import lightning as L
from spacetime.models.st_vq_vae import STVQVae, QuantizerType

import os
import wandb

In [2]:
if 'WANDB_API_KEY_SANDBOX' in os.environ:
    print("Using sandbox key")
    os.environ['WANDB_API_KEY'] = os.environ['WANDB_API_KEY_SANDBOX']

Using sandbox key


## Tokenizer Lightning module definition

We will use pytorch lightning to reduce boiler plate (there's a lot in previous notebooks, despite the centralized modules in `/src`)

In [3]:
class STVQVaeModule(L.LightningModule):
    def __init__(
        self,
        num_heads,
        d_model,
        num_layers,
        d_linear,
        codebook_size,
        codebook_dim,
        patch_size,
        frame_height,
        frame_width,
        num_frames,
        quantizer_type: QuantizerType = QuantizerType.EMA,
        num_linear_layers=2,
        num_groups=8,
        dropout=0.1,
        beta=0.1
    ):
        super().__init__()
        self.model = STVQVae(
            num_heads=num_heads,
            d_model=d_model,
            num_layers=num_layers,
            d_linear=d_linear,
            codebook_size=codebook_size,
            codebook_dim=codebook_dim,
            patch_size=patch_size,
            frame_height=frame_height,
            frame_width=frame_width,
            num_frames=num_frames,
            num_linear_layers=num_linear_layers,
            num_groups=num_groups,
            dropout=dropout,
            quantizer_type=quantizer_type
        )
        self.quantizer_type = quantizer_type
        self.beta = beta
        self.example_clip = None
        self.example_recon = None

        self.lpips_metric = lpips.LPIPS(net="vgg")
        self.lpips_metric.eval()
        for p in self.lpips_metric.parameters():
            p.requires_grad = False

    def forward(self, inputs):
        return self.model(inputs)

    def on_fit_start(self) -> None:
        super().on_fit_start()
        self.lpips_metric.to(self.device)
    
    def configure_optimizers(self):
      optimizer = torch.optim.AdamW(
          self.model.parameters(),
          lr=3e-4,
          betas=(0.9, 0.9),
          weight_decay=1e-4,
      )
      warmup_steps = 10_000

      def warmup_lambda(step):
          return min(1.0, (step + 1) / warmup_steps)

      scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_lambda)
      return {
          "optimizer": optimizer,
          "lr_scheduler": {
              "scheduler": scheduler,
              "interval": "step",
              "frequency": 1,
          },
      }

    def training_step(self, batch, batch_idx):
        x, _ = batch
        x_pred, z_e, z_quantized = self(x)
        recon_loss = torch.nn.functional.mse_loss(x_pred, x)
        commit_loss = torch.nn.functional.mse_loss(z_e, z_quantized.detach())
        codebook_loss = torch.nn.functional.mse_loss(z_quantized, z_e.detach()) if self.quantizer_type == QuantizerType.VANILLA else 0.0
        loss = recon_loss + (self.beta * commit_loss) + codebook_loss

        self._log_losses(loss, recon_loss, commit_loss, codebook_loss, is_training=True)

        if batch_idx == 0:
            self.example_clip = x[:1].detach().cpu()
            self.example_recon = x_pred[:1].detach().cpu()
        return loss

    def validation_step(self, batch, batch_idx):
        x, _ = batch
        x_pred, z_e, z_quantized = self(x)
        recon_loss = torch.nn.functional.mse_loss(x_pred, x)
        commit_loss = torch.nn.functional.mse_loss(z_e, z_quantized.detach())
        codebook_loss = torch.nn.functional.mse_loss(z_quantized, z_e.detach()) if self.quantizer_type == QuantizerType.VANILLA else 0.0
        loss = recon_loss + (self.beta * commit_loss) + codebook_loss

        self._log_losses(loss, recon_loss, commit_loss, codebook_loss, is_training=False)

        with torch.no_grad():
        # LPIPS expects inputs in [-1,1]; convert if in [0,1]
            B, C, F, H, W = x.shape
            to_lpips = lambda t: ((t * 2.0) - 1.0).reshape(B * F, C, H, W)
            lpips_val = self.lpips_metric(to_lpips(x_pred), to_lpips(x)).mean()
        self.log("val_lpips", lpips_val, prog_bar=False, logger=True)
        if wandb.run is not None:
            wandb.log({"val_lpips": lpips_val.item()}, step=self.global_step)
        return loss
    
    def on_validation_epoch_end(self):
        if self.example_clip is None or wandb.run is None:
            return
        clip = (self.example_clip.clamp(0, 1) * 255).to(torch.uint8)
        recon = (self.example_recon.clamp(0, 1) * 255).to(torch.uint8)
        video = torch.cat([clip, recon], dim=4)       # or dim=2/3, whichever you chose
        video = video.squeeze(0).permute(1, 0, 2, 3)  # (F, C, H, W)
        wandb.log(
            {
                "recon_video": wandb.Video(
                    video.squeeze(0), fps=4, format="mp4"
                )
            },
            step=self.global_step,
        )
        self.example_clip = None
        self.example_recon = None
    
    def _log_losses(self, loss, recon_loss, commit_loss, codebook_loss, is_training=True):
        prefix = "train" if is_training else "val"
        log_on_step = True if is_training else False
        log_on_epoch = True

        self.log(f"{prefix}_loss", loss, on_step=log_on_step, on_epoch=log_on_epoch, prog_bar=True, logger=True)
        self.log(f"{prefix}_recon_loss", recon_loss, on_step=log_on_step, on_epoch=log_on_epoch, prog_bar=False, logger=True)
        self.log(f"{prefix}_commit_loss", commit_loss, on_step=log_on_step, on_epoch=log_on_epoch, prog_bar=False, logger=True)
        if self.quantizer_type == QuantizerType.VANILLA:
            self.log(f"{prefix}_codebook_loss", codebook_loss, on_step=log_on_step, on_epoch=log_on_epoch, prog_bar=False, logger=True)

        if wandb.run is not None:
            log_dict = {
                f"{prefix}_loss": loss.item(),
                f"{prefix}_recon_loss": recon_loss.item(),
                f"{prefix}_commit_loss": commit_loss.item(),
            }
            if self.quantizer_type == QuantizerType.VANILLA:
                log_dict[f"{prefix}_codebook_loss"] = codebook_loss.item()
            wandb.log(log_dict, step=self.global_step)

    

## ProcGen: Heist Dataset Handling 

We will train our model on the "heist" environment from the [OpenAI Procgen Benchmark](https://github.com/openai/procgen). 
- Procgen is a suite of procedurally generated environments designed for benchmarking generalization in reinforcement learning agents.
- The Heist environment features randomly generated maze layouts on each episode, requiring the agent to navigate and collect keys to unlock safes.
- The dataset is generated by running the script in `src/spacetime/scripts/gen_procgen_heist.py`
- Our dataset consists of frame sequences and corresponding agent actions collected from Heist, preprocessed and saved as `.npz` shards for efficient loading.

In [4]:
from spacetime.utils.data import ProcgenShardDataset
from pathlib import Path

shard_dir = Path("../data/procgen_heist/shards")
shard_dir.mkdir(parents=True, exist_ok=True)

shard_dataset = ProcgenShardDataset(shard_dir, normalize=True)

KeyboardInterrupt: 

In [ ]:
from torch.utils.data import random_split

train_ratio = 0.8

train_size = int(train_ratio * len(shard_dataset))
val_size = len(shard_dataset) - train_size

train_dataset, val_dataset = random_split(
    shard_dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

In [ ]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=48,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=48,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
)


## Tokenizer Hyperparameters 

In [ ]:
params = {
    "num_heads": 4,  # starting with 4 heads paper uses 8 
    "d_model": 512,
    "num_layers": 4,  # starting with 4 layers paper uses 8
    "d_linear": 1536,
    "codebook_size": 1024,   # match latent model's num_discrete_actions
    "codebook_dim": 32,
    "patch_size": 8,  # starting with 8 paper uses 4 (but in a different type of setup)
    "frame_height": 64,
    "frame_width": 64,
    "num_frames": 16,
    "num_linear_layers": 2,
    "num_groups": 8,
    "dropout": 0.1,
    "max_epochs": 10,
    "precision": 16,
    "quantizer_type": "vanilla"
}

wandb.init(
    project="genie",
    name=f"tokenizer_{params['quantizer_type']}_{params['num_layers']}_heads{params['num_heads']}",
    config=params,
)

wandb: Currently logged in as: aryaman-pandya (aryaman-pandya-99) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
lightning_timesformer = STVQVaeModule(
    num_heads=params["num_heads"],
    d_model=params["d_model"],
    num_layers=params["num_layers"],
    d_linear=params["d_linear"],
    codebook_size=params["codebook_size"],
    codebook_dim=params["codebook_dim"],
    patch_size=params["patch_size"],
    frame_height=params["frame_height"],
    frame_width=params["frame_width"],
    num_frames=params["num_frames"],
    num_linear_layers=params["num_linear_layers"],
    num_groups=params["num_groups"],
    dropout=params["dropout"],
    quantizer_type=QuantizerType(params["quantizer_type"]),
)


#  wandb.watch(lightning_timesformer, log="gradients", log_freq=100)

"""

# use this trainer for sanity checks

trainer = L.Trainer(
    max_epochs=1,
    limit_train_batches=1,
    limit_val_batches=1,
    fast_dev_run=False,
)
"""

trainer = L.Trainer(max_epochs=20)

trainer.fit(model=lightning_timesformer, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

wandb.finish()

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Trainer will use only 1 of 4 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=4)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
💡 Tip: For seamless cloud uploads and versioning, try

Loading model from: /home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/lpips/weights/v0.1/vgg.pth


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA A100 80GB PCIe') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RAN

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7764f185b910>> (for post_run_cell), with arguments args (<ExecutionResult object at 7764f17d6470, execution_count=8 error_before_exec=None error_in_exec=1 info=<ExecutionInfo object at 7764f17d7040, raw_cell="lightning_timesformer = STVQVaeModule(
    num_hea.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Bcoder-vscode.coder.aks-prod-coder-swedencentral.azr.wayve.ai--aryamanpandya--4xa100.workspace/home/aryamanpandya/spacetime/nbs/tokenizer.ipynb#X14sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.s

BrokenPipeError: [Errno 32] Broken pipe

socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.


socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.s